# Importar Librerías y Configurar el Navegador

In [5]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from time import sleep
import csv
import os
from datetime import datetime
from random import randint
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import StaleElementReferenceException
import re

from selenium.common.exceptions import NoSuchElementException
from bs4 import BeautifulSoup, NavigableString

# ORQUESTADOR ----------------------------------------------------------------*


In [6]:
from scraping.extractor_twitter import iniciar_sesion, navegar_a_perfil, extraer_y_guardar_comentarios
#from utils.helpers import configurar_log

# Configuración de logs
#configurar_log()

# FUNCIÓN PRINCIPAL
try:
    # Configuración del navegador
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service)

    # Iniciar sesión
    iniciar_sesion(driver,user='juan_c.ortiz_b@uao.edu.co', pwd='3127916565', username='@OrtizBaron77043')

    # Navegar al perfil de @Tu_IMSS
    #navegar_a_perfil(driver,'https://x.com/Tu_IMSS')
    navegar_a_perfil(driver, "https://x.com/Tu_IMSS?f=live")
    print("¡Has iniciado sesión y estás en el perfil de @Tu_IMSS!")

    # Extraer y guardar comentarios de los primeros tweets
    #extraer_y_guardar_comentarios(driver, "comentarios_imss_urls_3.csv", max_tweets=20,n_scrolls=3,scroll_pause=2 )
    extraer_y_guardar_comentarios(driver, "comentarios_imss_urls_12.csv" )

except Exception as e:
    print("❌ Error en la ejecución:", e)

finally:
    # Cerrar el navegador
    if driver:
        driver.quit()


✔ Correo ingresado
ℹ No se pidió confirmación adicional de nombre de usuario.
✔ Contraseña ingresada
✔ Se hizo clic en 'Iniciar sesión'
¡Has iniciado sesión y estás en el perfil de @Tu_IMSS!
🔄 Scroll fijo #1/1
🔗 Capturadas 7 URLs de tweets (queríamos 20)

🔹 Procesando URL #1/7: https://x.com/Tu_IMSS/status/1950949577919729800

▶️  Abriendo tweet en nueva pestaña: https://x.com/Tu_IMSS/status/1950949577919729800
    🔄 Replies cargados: 0
    🔄 Replies cargados: 0
  🔍 Tamaño de page_source tras spam: 417446
  💡 Extraído: id=id_Perderse_en_una_historia_tambi replies=1
  👤 Autor original: IMSS  @Tu_IMSS
↪️ [guardar_comentarios] para tweet_id=id_Perderse_en_una_historia_tambi
🔎 oficiales: 0
🔎 spam_cells encontradas: 5, total spam artículos: 2
🔎 total candidatos tras unir: 1
→ auténticas tras filtrar y recortar a 1/1
🔄 Guardando respuesta auténtica #1
  🔙 Cerrando pestaña y volviendo…

🔹 Procesando URL #2/7: https://x.com/Tu_IMSS/status/1950723087319572658

▶️  Abriendo tweet en nueva pesta

In [ ]:
from scraping.extractor_twitter import iniciar_sesion, navegar_a_perfil, extraer_y_guardar_comentarios

# FUNCIÓN PRINCIPAL
try:
    # Configuración del navegador
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service)

    # Iniciar sesión
    iniciar_sesion(driver,user='jamoncayop@gmail.com', pwd='Adaptiv3@*', username='@Jaime1807816689')

    # Navegar al perfil de @Tu_IMSS
    navegar_a_perfil(driver, "https://x.com/Tu_IMSS?f=live")

    # Extraer y guardar comentarios de los primeros tweets
    extraer_y_guardar_comentarios(driver, "comentarios_imss_urls_3.csv", max_tweets=20,n_scrolls=3,scroll_pause=2 )

except Exception as e:
    print(" Error en la ejecución:", e)

finally:
    # Cerrar el navegador
    if driver:
        driver.quit()


In [ ]:



def procesar_tweet_por_url(driver, url, tweets_procesados, writer):
    print(f"\n▶️  Abriendo tweet en nueva pestaña: {url}")
    # 1) Abrir en pestaña nueva y cambiar contexto
    driver.execute_script("window.open(arguments[0], '_blank');", url)
    driver.switch_to.window(driver.window_handles[-1])
    WebDriverWait(driver, 15).until(lambda d: "/status/" in d.current_url)
    sleep(1)

    # 2) Reveal inicial de replies y posible spam
    for i in range(3):
        driver.execute_script("window.scrollBy(0, 800);")
        sleep(0.7)
    try:
        spam_btn = driver.find_element(
            By.XPATH, "//span[normalize-space(text())='Show probable spam']"
        )
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", spam_btn)
        sleep(0.5)
        spam_btn.click()
        sleep(1)
    except NoSuchElementException:
        pass

    # 3) Bucle hasta que ya no cargue más replies
    prev_count = -1
    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        sleep(1)

        # Capturamos el HTML y contamos replies (artículos menos el original)
        html = driver.page_source
        soup_tmp = BeautifulSoup(html, "html.parser")
        conv = soup_tmp.find("div", {
            "role": "region",
            "aria-label": re.compile(r"Timeline: Conversation")
        })
        loaded = (len(conv.find_all("article", {"data-testid": "tweet"})) - 1) if conv else 0

        print(f"    🔄 Replies cargados: {loaded}")
        if loaded == prev_count:
            break
        prev_count = loaded

    # 4) Ya con todo cargado, parseamos el resultado final
    print(f"  🔍 Tamaño de page_source tras spam: {len(driver.page_source)}")
    soup = BeautifulSoup(driver.page_source, "html.parser")

    # 5) Extraemos datos del tweet principal
    tweet_id, tweet_text, fecha_pub, replies, reposts, likes, views = extraer_datos_tweet(soup)
    print(f"  💡 Extraído: id={tweet_id} replies={replies}")

    # 6) Evitamos duplicados
    if tweet_id in tweets_procesados:
        print("  ⚠️ ya procesado, cierro y regreso.")
        driver.close()
        driver.switch_to.window(driver.window_handles[0])
        return
    tweets_procesados.add(tweet_id)

    # 7) Capturamos autor original
    owner_tag = soup.find("article", {"data-testid": "tweet"}) \
                    .find("div", {"data-testid": "User-Name"})
    tweet_owner_raw = owner_tag.get_text(" ", strip=True) if owner_tag else ""
    print(f"  👤 Autor original: {tweet_owner_raw}")

    # 8) Guardamos todos los comentarios ya cargados
    guardar_comentarios(
        tweet_id, tweet_text, soup, writer,
        fecha_pub, replies, reposts, likes, views,
        tweet_owner_raw
    )

    # 9) Cerramos pestaña y volvemos
    print("  🔙 Cerrando pestaña y volviendo…")
    driver.close()
    driver.switch_to.window(driver.window_handles[0])
    



In [ ]:
nombre = "camilo"